In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers datasets torch torchvision torchaudio \


from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers datasets torch torchvision torchaudio \
                matplotlib seaborn scikit-learn tqdm pandas numpy kaggle

import sys

import os
from pathlib import Path
import pandas as pd
import psutil
import numpy as np
from tqdm import tqdm



# Import only what we need
from core.cleaning import clean_text, SLANG_DICT  # Just this one import
from core.data_loaders import load_pan13, load_pan14

# Check memory
memory = psutil.virtual_memory()
print(f"Total RAM: {memory.total / 1e9:.2f} GB")
print(f"Available RAM: {memory.available / 1e9:.2f} GB")
print(f"RAM Usage: {memory.percent}%")

PAN13_PATHS = {
 'training': {
 'xml_dir': '/root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-training-corpus-2013-01-09/pan13-author-profiling-training-corpus-2013-01-09/en',
 'truth_file': '/root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-training-corpus-2013-01-09/pan13-author-profiling-training-corpus-2013-01-09/truth-en.txt'
    },
 'test': {
 'xml_dir': '/root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-test-corpus2-2013-04-29/pan13-author-profiling-test-corpus2-2013-04-29/en',
 'truth_file': '/root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-test-corpus2-2013-04-29/pan13-author-profiling-test-corpus2-2013-04-29/truth-en.txt'
    }
}

pan13_train = load_pan13(
    PAN13_PATHS['training']['xml_dir'],
    dataset_type='pan13_train',
    truth_file=PAN13_PATHS['training']['truth_file']
)

pan13_test = load_pan13(
    PAN13_PATHS['test']['xml_dir'],
    dataset_type='pan13_test',
    truth_file=PAN13_PATHS['test']['truth_file']
)

print(f"\nPAN13 Training: {len(pan13_train)} documents, {pan13_train['author_id'].nunique() if len(pan13_train) > 0 else 0} authors")
print(f"PAN13 Test: {len(pan13_test)} documents, {pan13_test['author_id'].nunique() if len(pan13_test) > 0 else 0} authors")


PAN14_PATHS = {
 'blogs': '/root/.cache/kagglehub/datasets/randmansour/pan-14/versions/1/pan14-author-profiling-training-corpus-2014-04-16/pan14-author-profiling-training-corpus-english-blogs-2014-04-16/mnt/nfs/tira/data/pan14-training-corpora-truth/pan14-author-profiling-training-corpus-english-blogs-2014-04-16',
 'reviews': '/root/.cache/kagglehub/datasets/randmansour/pan-14/versions/1/pan14-author-profiling-training-corpus-2014-04-16/pan14-author-profiling-training-corpus-english-reviews-2014-04-16/mnt/nfs/tira/data/pan14-training-corpora-truth/pan14-author-profiling-training-corpus-english-reviews-2014-04-16',
 'socialmedia': '/root/.cache/kagglehub/datasets/randmansour/pan-14/versions/1/pan14-author-profiling-training-corpus-2014-04-16/pan14-author-profiling-training-corpus-english-socialmedia-2014-04-16/mnt/nfs/tira/data/pan14-training-corpora-truth/pan14-author-profiling-training-corpus-english-socialmedia-2014-04-16',
 'twitter': '/root/.cache/kagglehub/datasets/randmansour/pan-14/versions/1/pan14-author-profiling-training-corpus-2014-04-16/pan14-author-profiling-training-corpus-english-twitter-2014-04-16/mnt/nfs/tira/data/pan14-training-corpora-truth/pan14-author-profiling-training-corpus-english-twitter-2014-04-16'
}

TRUTH_PATHS = {
 'blogs': f"{PAN14_PATHS['blogs']}/truth.txt",
 'reviews': f"{PAN14_PATHS['reviews']}/truth.txt",
 'socialmedia': f"{PAN14_PATHS['socialmedia']}/truth.txt",
 'twitter': f"{PAN14_PATHS['twitter']}/truth.txt"
}
pan14_dfs = []
for dtype, path in PAN14_PATHS.items():
    df = load_pan14(path, dataset_type=dtype, truth_file=TRUTH_PATHS.get(dtype))
    pan14_dfs.append(df)



pan14_df = pd.concat(pan14_dfs, ignore_index=True) if pan14_dfs else pd.DataFrame()
print(f"\nTotal PAN14: {len(pan14_df)} documents, {pan14_df['author_id'].nunique() if len(pan14_df) > 0 else 0} authors")


len(pan14_df['author_id'].unique())
12053

pan13_test_df = pan13_test[['text', 'age_group', 'author_id', 'dataset_type']].copy()

train_dfs = []
train_dfs.append(pan13_train[['text', 'age_group', 'author_id', 'dataset_type']].copy())
train_dfs.append(pan14_df[['text', 'age_group', 'author_id', 'dataset_type']].copy())

all_train_data = pd.concat(train_dfs, ignore_index=True)
print(f"\n Combined training data: {len(all_train_data)} documents")
print(f" Unique authors: {all_train_data['author_id'].nunique()}")

print("\n Documents per dataset (before cleaning):")
print(all_train_data['dataset_type'].value_counts())

print("\n Age group distribution (BEFORE unification):")
print(all_train_data['age_group'].value_counts())

print(f"\n Cleaning {len(all_train_data)} training documents...")
train_clean = all_train_data.copy()

tqdm.pandas(desc="Cleaning training documents")
train_clean['text'] = train_clean['text'].progress_apply(
    lambda x: clean_text(x, aggressive=True, slang_dict=SLANG_DICT)
)

# Filter out empty texts
original_len = len(train_clean)
train_clean = train_clean[train_clean['text'].str.len() > 0].reset_index(drop=True)

print(f"\n Cleaned training data: {train_clean.shape}")
print(f" Rows removed: {original_len - len(train_clean)} ({((original_len - len(train_clean))/original_len*100):.1f}%)")

# Clean test data
print(f"\n Cleaning {len(pan13_test_df)} test documents...")
test_clean = pan13_test_df.copy()

tqdm.pandas(desc="Cleaning test documents")
test_clean['text'] = test_clean['text'].progress_apply(
    lambda x: clean_text(x, aggressive=True, slang_dict=SLANG_DICT)
)

# Filter out empty texts
original_test_len = len(test_clean)
test_clean = test_clean[test_clean['text'].str.len() > 0].reset_index(drop=True)

print(f"\n Cleaned test data: {test_clean.shape}")
print(f" Rows removed: {original_test_len - len(test_clean)} ({((original_test_len - len(test_clean))/original_test_len*100):.1f}%)")

# REMOVE DUPLICATES AND NULLS (TRAIN & TEST)

print("\n" + "="*50)
print("REMOVING DUPLICATES AND NULLS")
print("="*50)

# Function to clean dataset
def clean_dataset(df, dataset_name, remove_author_duplicates=True):
    print(f"\n Cleaning {dataset_name}: {len(df)} rows")

    # 1. Remove duplicate texts
    before = len(df)
    df = df.drop_duplicates(subset=['text'])
    print(f" Removed {before - len(df)} duplicate text rows")

    # 2. Remove duplicate author-text combinations (if author_id exists)
    if remove_author_duplicates and 'author_id' in df.columns:
        before = len(df)
        df = df.drop_duplicates(subset=['author_id', 'text'])
        print(f" Removed {before - len(df)} duplicate author-text combinations")

    # 3. Remove nulls
    null_cols = ['text', 'age_group']
    if 'author_id' in df.columns:
        null_cols.append('author_id')

    print(f"\nNull values before removal:")
    for col in null_cols:
        if col in df.columns:
            print(f" {col}: {df[col].isnull().sum()}")

    df = df.dropna(subset=null_cols)
    print(f" Removed null rows")

    # 4. Remove whitespace-only texts
    before = len(df)
    df = df[df['text'].str.strip().str.len() > 0]
    print(f" Removed {before - len(df)} whitespace-only texts")

    # 5. Remove extremely long texts (prevents BERT token overflow)
    MAX_TEXT_LEN = 10000
    before = len(df)
    df = df[df['text'].str.len() <= MAX_TEXT_LEN]
    print(f" Removed {before - len(df)} extremely long texts (> {MAX_TEXT_LEN} chars)")

    # 6. Final quality check
    print(f"\n Final {dataset_name}: {len(df)} samples")
    print(f" Remaining duplicates: {df.duplicated(subset=['text']).sum()}")
    print(f" Remaining nulls: {df[null_cols].isnull().sum().sum()}")

    return df

# Apply to both train and test
train_clean = clean_dataset(train_clean, "TRAIN", remove_author_duplicates=True)
test_clean = clean_dataset(test_clean, "TEST", remove_author_duplicates=False)  # Test may not have author_id

# Add length column
train_clean['length'] = train_clean['text'].str.split().str.len()
test_clean['length'] = test_clean['text'].str.split().str.len()

# Set minimum words threshold
MIN_WORDS = 10

# Filter training data
before_train = len(train_clean)
train_clean = train_clean[train_clean['length'] >= MIN_WORDS].reset_index(drop=True)
print(f"Training: Removed {before_train - len(train_clean)} short conversations (<{MIN_WORDS} words)")

# Filter test data
before_test = len(test_clean)
test_clean = test_clean[test_clean['length'] >= MIN_WORDS].reset_index(drop=True)
print(f"Test: Removed {before_test - len(test_clean)} short conversations (<{MIN_WORDS} words)")

# Check length statistics
print(f"\n Text length statistics:")
print(f"Training - Min: {train_clean['length'].min()}, Max: {train_clean['length'].max()}, Mean: {train_clean['length'].mean():.1f}")
print(f"Test - Min: {test_clean['length'].min()}, Max: {test_clean['length'].max()}, Mean: {test_clean['length'].mean():.1f}")

def unify_age_groups(age_group):
 """
    Unified age group mapping - only Minor and Adult
 """
    # Handle NaN or None
    if pd.isna(age_group):
        return np.nan

    age_str = str(age_group).strip()

    # Handle annotated formats like "30s:::pedophile", "20s:::sex"
    if ':::' in age_str:
        age_str = age_str.split(':::')[0]

    # PAN14 decade formats
    if age_str == '10s':
        return 'Minor'
    elif age_str in ['20s', '30s']:
        return 'Adult'

    # PAN13 and PAN14 range formats
    elif age_str == '13-17':
        return 'Minor'
    elif age_str in ['18-24', '25-34', '35-49', '50-64', '50-65', '65-xx']:
        return 'Adult'
    else:
        # Only warn about truly unknown formats
        print(f" Warning: Unknown age format: '{age_group}'")
        return np.nan

[ ]

# Save original for debugging
test_clean['age_group_original'] = test_clean['age_group']

# Apply unification to test data
test_clean['age_group'] = test_clean['age_group_original'].apply(unify_age_groups)

print("\n TEST DATA BEFORE UNIFICATION:")
print(test_clean['age_group_original'].value_counts())

print("\n TEST DATA AFTER UNIFICATION:")
print(test_clean['age_group'].value_counts())

# Show mapping for test data
print("\n TEST DATA MAPPING SUMMARY:")
test_mapping = pd.crosstab(
    test_clean['age_group_original'],
    test_clean['age_group'],
    margins=True
)
print(test_mapping)



train_clean['age_group'] = train_clean['age_group'].apply(unify_age_groups)

before_train = len(train_clean)
before_test = len(test_clean)

train_clean = train_clean.dropna(subset=['age_group']).reset_index(drop=True)
test_clean = test_clean.dropna(subset=['age_group']).reset_index(drop=True)

print(f"\n Removed invalid rows:")
print(f"Train: {before_train - len(train_clean)} removed")
print(f"Test: {before_test - len(test_clean)} removed")


# Get unique authors
authors = train_clean['author_id'].unique()
print(f"Total unique authors: {len(authors)}")

# Get author-level labels for stratification
author_labels = train_clean.groupby('author_id').agg({
 'age_group': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown',  # Use most common age
 'author_id': 'count' # Count documents per author
}).rename(columns={'author_id': 'doc_count'}).reset_index()

print(f"\nAuthor document count distribution:")
print(author_labels['doc_count'].describe())

# Check for authors with multiple age groups (potential data quality issue)
authors_with_multiple_ages = train_clean.groupby('author_id')['age_group'].nunique()
multi_age_authors = authors_with_multiple_ages[authors_with_multiple_ages > 1]
print(f"\n Authors with inconsistent age labels: {len(multi_age_authors)} ({len(multi_age_authors)/len(authors)*100:.1f}%)")

# Split authors into train/val (80/20) stratified by age group
from sklearn.model_selection import train_test_split

train_authors, val_authors = train_test_split(
    author_labels['author_id'].values,
    test_size=0.2,
    random_state=42,
    stratify=author_labels['age_group']  # Stratify by age group
)

print(f"\nTrain authors: {len(train_authors)}")
print(f"Validation authors: {len(val_authors)}")

# Create train and validation sets
final_train = train_clean[train_clean['author_id'].isin(train_authors)].reset_index(drop=True)
final_val = train_clean[train_clean['author_id'].isin(val_authors)].reset_index(drop=True)


print(f"\n Final dataset sizes:")
print(f" Train: {len(final_train)} documents, {final_train['author_id'].nunique()} authors")
print(f" Validation: {len(final_val)} documents, {final_val['author_id'].nunique()} authors")
print(f" Test: {len(test_clean)} documents, {test_clean['author_id'].nunique()} authors")

# Check label distribution across splits
print("\n Age group distribution across splits:")
print("\nTrain:")
print(final_train['age_group'].value_counts(normalize=True))
print("\nValidation:")
print(final_val['age_group'].value_counts(normalize=True))
print("\nTest:")
print(test_clean['age_group'].value_counts(normalize=True))



if len(final_train) > 0:
    sample_indices = np.random.choice(len(final_train), size=min(3, len(final_train)), replace=False)

    for idx in sample_indices:
        print("\n" + "-"*80)
        print(f"TRAIN EXAMPLE (Age: {final_train['age_group'].iloc[idx]}")
        print(f"Source: {final_train['dataset_type'].iloc[idx]}")
        print(f"Author: {final_train['author_id'].iloc[idx]}")
        print("-"*80)

        cleaned_text = final_train['text'].iloc[idx]
        print(f"\n CLEANED TEXT:")
        print(cleaned_text[:500] + "..." if len(cleaned_text) > 500 else cleaned_text)


# Create directory in Drive
DATA_PATH = os.getenv('DATA_PATH')
os.makedirs(DATA_PATH, exist_ok=True)

# Save train split to Drive
train_file = f'{DATA_PATH}/author_train_clean.csv'
final_train.to_csv(train_file, index=False)
print(f" Saved train data to {train_file}")
print(f" Shape: {final_train.shape}")

# Save validation split to Drive
val_file = f'{DATA_PATH}/author_val_clean.csv'
final_val.to_csv(val_file, index=False)
print(f" Saved validation data to {val_file}")
print(f" Shape: {final_val.shape}")

# Save test split to Drive
test_file = f'{DATA_PATH}/author_test_clean.csv'
test_clean.to_csv(test_file, index=False)
print(f" Saved test data to {test_file}")
print(f" Shape: {test_clean.shape}")


print(f"\nFiles saved in Google Drive:")
print(f" - {DATA_PATH}/author_train_clean.csv")
print(f" - {DATA_PATH}/author_val_clean.csv")
print(f" - {DATA_PATH}/author_test_clean.csv")


import sys

import os
import json
from pathlib import Path

import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW

from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import kagglehub

from core.training import ChatDataset, train_epoch, evaluate, save_model


randmansour_pan_14_path = kagglehub.dataset_download('randmansour/pan-14')
randmansour_pan_13_path = kagglehub.dataset_download('randmansour/pan-13')
print(randmansour_pan_14_path)
print(randmansour_pan_13_path)

In [ ]:
from pathlib import Path
import pandas as pd
import psutil
import numpy as np
from tqdm import tqdm



# Import only what we need
from core.cleaning import clean_text, SLANG_DICT  # Just this one import
from core.data_loaders import load_pan13, load_pan14

# Check memory
memory = psutil.virtual_memory()
print(f"Total RAM: {memory.total / 1e9:.2f} GB")
print(f"Available RAM: {memory.available / 1e9:.2f} GB")
print(f"RAM Usage: {memory.percent}%")

## Loading Pan13 and Pan14

In [ ]:
!ls /root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1

In [ ]:
!ls /root/.cache/kagglehub/datasets/randmansour/pan-14/versions/1/pan14-author-profiling-training-corpus-2014-04-16/pan14-author-profiling-training-corpus-english-blogs-2014-04-16/mnt/nfs/tira/data

In [ ]:

PAN13_PATHS = {
 'training': {
 'xml_dir': '/root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-training-corpus-2013-01-09/pan13-author-profiling-training-corpus-2013-01-09/en',
 'truth_file': '/root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-training-corpus-2013-01-09/pan13-author-profiling-training-corpus-2013-01-09/truth-en.txt'
    },
 'test': {
 'xml_dir': '/root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-test-corpus2-2013-04-29/pan13-author-profiling-test-corpus2-2013-04-29/en',
 'truth_file': '/root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-test-corpus2-2013-04-29/pan13-author-profiling-test-corpus2-2013-04-29/truth-en.txt'
    }
}

pan13_train = load_pan13(
    PAN13_PATHS['training']['xml_dir'],
    dataset_type='pan13_train',
    truth_file=PAN13_PATHS['training']['truth_file']
)

pan13_test = load_pan13(
    PAN13_PATHS['test']['xml_dir'],
    dataset_type='pan13_test',
    truth_file=PAN13_PATHS['test']['truth_file']
)

print(f"\nPAN13 Training: {len(pan13_train)} documents, {pan13_train['author_id'].nunique() if len(pan13_train) > 0 else 0} authors")
print(f"PAN13 Test: {len(pan13_test)} documents, {pan13_test['author_id'].nunique() if len(pan13_test) > 0 else 0} authors")


PAN14_PATHS = {
 'blogs': '/root/.cache/kagglehub/datasets/randmansour/pan-14/versions/1/pan14-author-profiling-training-corpus-2014-04-16/pan14-author-profiling-training-corpus-english-blogs-2014-04-16/mnt/nfs/tira/data/pan14-training-corpora-truth/pan14-author-profiling-training-corpus-english-blogs-2014-04-16',
 'reviews': '/root/.cache/kagglehub/datasets/randmansour/pan-14/versions/1/pan14-author-profiling-training-corpus-2014-04-16/pan14-author-profiling-training-corpus-english-reviews-2014-04-16/mnt/nfs/tira/data/pan14-training-corpora-truth/pan14-author-profiling-training-corpus-english-reviews-2014-04-16',
 'socialmedia': '/root/.cache/kagglehub/datasets/randmansour/pan-14/versions/1/pan14-author-profiling-training-corpus-2014-04-16/pan14-author-profiling-training-corpus-english-socialmedia-2014-04-16/mnt/nfs/tira/data/pan14-training-corpora-truth/pan14-author-profiling-training-corpus-english-socialmedia-2014-04-16',
 'twitter': '/root/.cache/kagglehub/datasets/randmansour/pan-14/versions/1/pan14-author-profiling-training-corpus-2014-04-16/pan14-author-profiling-training-corpus-english-twitter-2014-04-16/mnt/nfs/tira/data/pan14-training-corpora-truth/pan14-author-profiling-training-corpus-english-twitter-2014-04-16'
}

TRUTH_PATHS = {
 'blogs': f"{PAN14_PATHS['blogs']}/truth.txt",
 'reviews': f"{PAN14_PATHS['reviews']}/truth.txt",
 'socialmedia': f"{PAN14_PATHS['socialmedia']}/truth.txt",
 'twitter': f"{PAN14_PATHS['twitter']}/truth.txt"
}
pan14_dfs = []
for dtype, path in PAN14_PATHS.items():
    df = load_pan14(path, dataset_type=dtype, truth_file=TRUTH_PATHS.get(dtype))
    pan14_dfs.append(df)



pan14_df = pd.concat(pan14_dfs, ignore_index=True) if pan14_dfs else pd.DataFrame()
print(f"\nTotal PAN14: {len(pan14_df)} documents, {pan14_df['author_id'].nunique() if len(pan14_df) > 0 else 0} authors")


In [ ]:
len(pan14_df['author_id'].unique())

In [ ]:
pan13_test_df = pan13_test[['text', 'age_group', 'author_id', 'dataset_type']].copy()

train_dfs = []
train_dfs.append(pan13_train[['text', 'age_group', 'author_id', 'dataset_type']].copy())
train_dfs.append(pan14_df[['text', 'age_group', 'author_id', 'dataset_type']].copy())

all_train_data = pd.concat(train_dfs, ignore_index=True)
print(f"\n Combined training data: {len(all_train_data)} documents")
print(f" Unique authors: {all_train_data['author_id'].nunique()}")

print("\n Documents per dataset (before cleaning):")
print(all_train_data['dataset_type'].value_counts())

print("\n Age group distribution (BEFORE unification):")
print(all_train_data['age_group'].value_counts())


UNIFYING AGE GROUPS ACROSS DATASETS

In [ ]:
print(f"\n Cleaning {len(all_train_data)} training documents...")
train_clean = all_train_data.copy()

tqdm.pandas(desc="Cleaning training documents")
train_clean['text'] = train_clean['text'].progress_apply(
    lambda x: clean_text(x, aggressive=True, slang_dict=SLANG_DICT)
)

# Filter out empty texts
original_len = len(train_clean)
train_clean = train_clean[train_clean['text'].str.len() > 0].reset_index(drop=True)

print(f"\n Cleaned training data: {train_clean.shape}")
print(f" Rows removed: {original_len - len(train_clean)} ({((original_len - len(train_clean))/original_len*100):.1f}%)")

# Clean test data
print(f"\n Cleaning {len(pan13_test_df)} test documents...")
test_clean = pan13_test_df.copy()

tqdm.pandas(desc="Cleaning test documents")
test_clean['text'] = test_clean['text'].progress_apply(
    lambda x: clean_text(x, aggressive=True, slang_dict=SLANG_DICT)
)

# Filter out empty texts
original_test_len = len(test_clean)
test_clean = test_clean[test_clean['text'].str.len() > 0].reset_index(drop=True)

print(f"\n Cleaned test data: {test_clean.shape}")
print(f" Rows removed: {original_test_len - len(test_clean)} ({((original_test_len - len(test_clean))/original_test_len*100):.1f}%)")

# REMOVE DUPLICATES AND NULLS (TRAIN & TEST)

print("\n" + "="*50)
print("REMOVING DUPLICATES AND NULLS")
print("="*50)

# Function to clean dataset
def clean_dataset(df, dataset_name, remove_author_duplicates=True):
    print(f"\n Cleaning {dataset_name}: {len(df)} rows")

    # 1. Remove duplicate texts
    before = len(df)
    df = df.drop_duplicates(subset=['text'])
    print(f" Removed {before - len(df)} duplicate text rows")

    # 2. Remove duplicate author-text combinations (if author_id exists)
    if remove_author_duplicates and 'author_id' in df.columns:
        before = len(df)
        df = df.drop_duplicates(subset=['author_id', 'text'])
        print(f" Removed {before - len(df)} duplicate author-text combinations")

    # 3. Remove nulls
    null_cols = ['text', 'age_group']
    if 'author_id' in df.columns:
        null_cols.append('author_id')

    print(f"\nNull values before removal:")
    for col in null_cols:
        if col in df.columns:
            print(f" {col}: {df[col].isnull().sum()}")

    df = df.dropna(subset=null_cols)
    print(f" Removed null rows")

    # 4. Remove whitespace-only texts
    before = len(df)
    df = df[df['text'].str.strip().str.len() > 0]
    print(f" Removed {before - len(df)} whitespace-only texts")

    # 5. Remove extremely long texts (prevents BERT token overflow)
    MAX_TEXT_LEN = 10000
    before = len(df)
    df = df[df['text'].str.len() <= MAX_TEXT_LEN]
    print(f" Removed {before - len(df)} extremely long texts (> {MAX_TEXT_LEN} chars)")

    # 6. Final quality check
    print(f"\n Final {dataset_name}: {len(df)} samples")
    print(f" Remaining duplicates: {df.duplicated(subset=['text']).sum()}")
    print(f" Remaining nulls: {df[null_cols].isnull().sum().sum()}")

    return df

# Apply to both train and test
train_clean = clean_dataset(train_clean, "TRAIN", remove_author_duplicates=True)
test_clean = clean_dataset(test_clean, "TEST", remove_author_duplicates=False)  # Test may not have author_id


In [ ]:
# Add length column
train_clean['length'] = train_clean['text'].str.split().str.len()
test_clean['length'] = test_clean['text'].str.split().str.len()

# Set minimum words threshold
MIN_WORDS = 10

# Filter training data
before_train = len(train_clean)
train_clean = train_clean[train_clean['length'] >= MIN_WORDS].reset_index(drop=True)
print(f"Training: Removed {before_train - len(train_clean)} short conversations (<{MIN_WORDS} words)")

# Filter test data
before_test = len(test_clean)
test_clean = test_clean[test_clean['length'] >= MIN_WORDS].reset_index(drop=True)
print(f"Test: Removed {before_test - len(test_clean)} short conversations (<{MIN_WORDS} words)")

# Check length statistics
print(f"\n Text length statistics:")
print(f"Training - Min: {train_clean['length'].min()}, Max: {train_clean['length'].max()}, Mean: {train_clean['length'].mean():.1f}")
print(f"Test - Min: {test_clean['length'].min()}, Max: {test_clean['length'].max()}, Mean: {test_clean['length'].mean():.1f}")

In [ ]:
def unify_age_groups(age_group):
 """
    Unified age group mapping - only Minor and Adult
 """
    # Handle NaN or None
    if pd.isna(age_group):
        return np.nan

    age_str = str(age_group).strip()

    # Handle annotated formats like "30s:::pedophile", "20s:::sex"
    if ':::' in age_str:
        age_str = age_str.split(':::')[0]

    # PAN14 decade formats
    if age_str == '10s':
        return 'Minor'
    elif age_str in ['20s', '30s']:
        return 'Adult'

    # PAN13 and PAN14 range formats
    elif age_str == '13-17':
        return 'Minor'
    elif age_str in ['18-24', '25-34', '35-49', '50-64', '50-65', '65-xx']:
        return 'Adult'
    else:
        # Only warn about truly unknown formats
        print(f" Warning: Unknown age format: '{age_group}'")
        return np.nan

In [ ]:

# Save original for debugging
test_clean['age_group_original'] = test_clean['age_group']

# Apply unification to test data
test_clean['age_group'] = test_clean['age_group_original'].apply(unify_age_groups)

print("\n TEST DATA BEFORE UNIFICATION:")
print(test_clean['age_group_original'].value_counts())

print("\n TEST DATA AFTER UNIFICATION:")
print(test_clean['age_group'].value_counts())

# Show mapping for test data
print("\n TEST DATA MAPPING SUMMARY:")
test_mapping = pd.crosstab(
    test_clean['age_group_original'],
    test_clean['age_group'],
    margins=True
)
print(test_mapping)



train_clean['age_group'] = train_clean['age_group'].apply(unify_age_groups)

before_train = len(train_clean)
before_test = len(test_clean)

train_clean = train_clean.dropna(subset=['age_group']).reset_index(drop=True)
test_clean = test_clean.dropna(subset=['age_group']).reset_index(drop=True)

print(f"\n Removed invalid rows:")
print(f"Train: {before_train - len(train_clean)} removed")
print(f"Test: {before_test - len(test_clean)} removed")

In [ ]:
# Get unique authors
authors = train_clean['author_id'].unique()
print(f"Total unique authors: {len(authors)}")

# Get author-level labels for stratification
author_labels = train_clean.groupby('author_id').agg({
 'age_group': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown',  # Use most common age
 'author_id': 'count' # Count documents per author
}).rename(columns={'author_id': 'doc_count'}).reset_index()

print(f"\nAuthor document count distribution:")
print(author_labels['doc_count'].describe())

# Check for authors with multiple age groups (potential data quality issue)
authors_with_multiple_ages = train_clean.groupby('author_id')['age_group'].nunique()
multi_age_authors = authors_with_multiple_ages[authors_with_multiple_ages > 1]
print(f"\n Authors with inconsistent age labels: {len(multi_age_authors)} ({len(multi_age_authors)/len(authors)*100:.1f}%)")

# Split authors into train/val (80/20) stratified by age group
from sklearn.model_selection import train_test_split

train_authors, val_authors = train_test_split(
    author_labels['author_id'].values,
    test_size=0.2,
    random_state=42,
    stratify=author_labels['age_group']  # Stratify by age group
)

print(f"\nTrain authors: {len(train_authors)}")
print(f"Validation authors: {len(val_authors)}")

# Create train and validation sets
final_train = train_clean[train_clean['author_id'].isin(train_authors)].reset_index(drop=True)
final_val = train_clean[train_clean['author_id'].isin(val_authors)].reset_index(drop=True)


print(f"\n Final dataset sizes:")
print(f" Train: {len(final_train)} documents, {final_train['author_id'].nunique()} authors")
print(f" Validation: {len(final_val)} documents, {final_val['author_id'].nunique()} authors")
print(f" Test: {len(test_clean)} documents, {test_clean['author_id'].nunique()} authors")

# Check label distribution across splits
print("\n Age group distribution across splits:")
print("\nTrain:")
print(final_train['age_group'].value_counts(normalize=True))
print("\nValidation:")
print(final_val['age_group'].value_counts(normalize=True))
print("\nTest:")
print(test_clean['age_group'].value_counts(normalize=True))


In [ ]:

if len(final_train) > 0:
    sample_indices = np.random.choice(len(final_train), size=min(3, len(final_train)), replace=False)

    for idx in sample_indices:
        print("\n" + "-"*80)
        print(f"TRAIN EXAMPLE (Age: {final_train['age_group'].iloc[idx]}")
        print(f"Source: {final_train['dataset_type'].iloc[idx]}")
        print(f"Author: {final_train['author_id'].iloc[idx]}")
        print("-"*80)

        cleaned_text = final_train['text'].iloc[idx]
        print(f"\n CLEANED TEXT:")
        print(cleaned_text[:500] + "..." if len(cleaned_text) > 500 else cleaned_text)


In [ ]:


DATA_PATH = os.getenv('DATA_PATH')
os.makedirs(DATA_PATH, exist_ok=True)

# Save train split to Drive
train_file = f'{DATA_PATH}/author_train_clean.csv'
final_train.to_csv(train_file, index=False)
print(f" Saved train data to {train_file}")
print(f" Shape: {final_train.shape}")

# Save validation split to Drive
val_file = f'{DATA_PATH}/author_val_clean.csv'
final_val.to_csv(val_file, index=False)
print(f" Saved validation data to {val_file}")
print(f" Shape: {final_val.shape}")

# Save test split to Drive
test_file = f'{DATA_PATH}/author_test_clean.csv'
test_clean.to_csv(test_file, index=False)
print(f" Saved test data to {test_file}")
print(f" Shape: {test_clean.shape}")


print(f"\nFiles saved in Google Drive:")
print(f" - {DATA_PATH}/author_train_clean.csv")
print(f" - {DATA_PATH}/author_val_clean.csv")
print(f" - {DATA_PATH}/author_test_clean.csv")